# Plaka Tespit Modeli — YOLOv8n Fine-Tuning

**Amaç:** Roboflow Universe'den fork edilen Türk plaka dataset'i üzerinde YOLOv8n fine-tuning yaparak `plate.pt` weight'i üretmek.

**Çıktılar (tez için):**
- `weights/plate.pt` — local pipeline'a entegre edilecek model
- `thesis_figures/*.pdf` — yayın kalitesinde grafikler (loss/mAP eğrileri, FPS histogramı, örnek prediction'lar, failure case'ler)
- `metrics/thesis_metrics.json` — tez tablolarına direkt kopyalanacak metrik değerleri

**Real-time hedefi:** YOLOv8n ile 30+ FPS on T4 / RTX 3060.

**Augmentation stratejisi:** Roboflow tarafında augmentation YOK (free tier'a sığsın diye); tüm augmentation YOLOv8 training-time online (mosaic, mixup, hsv, rotation, scale). Flip kapalı — plaka asimetrik.

## 1. Ortam Kurulumu

In [ ]:
import torch, os, sys
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA: {torch.version.cuda}')
else:
    print('UYARI: GPU yok, eğitim CPU\'da çok yavaş olur. Runtime → Change runtime type → T4 GPU seç.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

SAVE_DIR = '/content/drive/MyDrive/tez_models/plate_detector'
FIGURES_DIR = f'{SAVE_DIR}/thesis_figures'
METRICS_DIR = f'{SAVE_DIR}/metrics'
WEIGHTS_DIR = f'{SAVE_DIR}/weights'
for d in [SAVE_DIR, FIGURES_DIR, METRICS_DIR, WEIGHTS_DIR]:
    os.makedirs(d, exist_ok=True)
print(f'Çıktı klasörü: {SAVE_DIR}')

In [ ]:
!pip install -q ultralytics roboflow wandb seaborn

## 2. Roboflow Dataset İndirme

**API key güvenliği:** Colab'ın sol panelinden 🔑 (Secrets) → `ROBOFLOW_API_KEY` adıyla key'i ekle, **"Notebook access"** anahtarını aç. Aşağıdaki kod key'i secret'tan okur, kod içine yazılmaz.

Eğer key'i yeniden oluşturmadıysan: Roboflow → Settings → API Keys → Revoke + Regenerate.

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

ROBOFLOW_WORKSPACE = 'sertas-workspace'
ROBOFLOW_PROJECT   = 'tr-plaka-grpln'
ROBOFLOW_VERSION   = 1

api_key = userdata.get('ROBOFLOW_API_KEY')
assert api_key, 'ROBOFLOW_API_KEY Colab Secrets\'a eklenmemiş'

rf = Roboflow(api_key=api_key)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)
dataset = version.download('yolov8')

DATASET_DIR = dataset.location
DATA_YAML = f'{DATASET_DIR}/data.yaml'
print(f'\nDataset: {DATASET_DIR}')
!cat $DATA_YAML

In [ ]:
import matplotlib.pyplot as plt
import cv2, random

split_counts = {}
for split in ['train', 'valid', 'test']:
    img_dir = f'{DATASET_DIR}/{split}/images'
    lbl_dir = f'{DATASET_DIR}/{split}/labels'
    if os.path.exists(img_dir):
        n_img = len(os.listdir(img_dir))
        n_lbl = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0
        split_counts[split] = {'images': n_img, 'labels': n_lbl}
        print(f'{split:6s}: {n_img:5d} images, {n_lbl:5d} labels')

train_imgs = sorted(os.listdir(f'{DATASET_DIR}/train/images'))
samples = random.sample(train_imgs, min(9, len(train_imgs)))
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
for ax, name in zip(axes.flatten(), samples):
    img = cv2.imread(f'{DATASET_DIR}/train/images/{name}')
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    lbl_path = f'{DATASET_DIR}/train/labels/{name.rsplit(".",1)[0]}.txt'
    if os.path.exists(lbl_path):
        h, w = img.shape[:2]
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cx, cy, bw, bh = map(float, parts[1:5])
                    x1 = int((cx - bw/2) * w); y1 = int((cy - bh/2) * h)
                    x2 = int((cx + bw/2) * w); y2 = int((cy + bh/2) * h)
                    cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)
    ax.imshow(img); ax.axis('off')
plt.suptitle('Dataset Örnekleri (train, ground-truth bbox)', fontsize=14)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/dataset_samples.pdf', bbox_inches='tight', dpi=200)
plt.savefig(f'{FIGURES_DIR}/dataset_samples.png', bbox_inches='tight', dpi=150)
plt.show()

## 3. Eğitim — YOLOv8n Fine-Tuning

**Augmentation gerekçeleri (tez bölümü):**
- `mosaic=1.0` — 4 görüntü tek karede, küçük plaka detection için altın standart
- `mixup=0.1` — hafif mixup, overfit'i azaltır
- `degrees=10.0` — kameranın ±10° eğikliğini simüle eder
- `hsv_v=0.4`, `hsv_s=0.5` — gündüz/gece, parlak/loş varyasyonu
- `fliplr=0.0`, `flipud=0.0` — **kapalı**, plaka asimetrik ("34" → "43" olur)
- `cos_lr=True` — cosine learning rate schedule

In [ ]:
USE_WANDB = False
if USE_WANDB:
    import wandb
    wandb.login()
    wandb.init(project='plate-detector-thesis', name='yolov8n_v1',
               config={'model':'yolov8n', 'epochs':80, 'imgsz':640, 'batch':32})

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=DATA_YAML,
    epochs=80,
    imgsz=640,
    batch=32,
    patience=20,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    augment=True,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.0,
    flipud=0.0,
    project=SAVE_DIR,
    name='train',
    exist_ok=True,
    save=True,
    plots=True,
    seed=42,
)

TRAIN_DIR = f'{SAVE_DIR}/train'
BEST_PT = f'{TRAIN_DIR}/weights/best.pt'
print(f'\n✓ Eğitim tamamlandı.\nBest weight: {BEST_PT}')

## 4. Test Set Değerlendirme

In [ ]:
best_model = YOLO(BEST_PT)
metrics = best_model.val(data=DATA_YAML, split='test', plots=True,
                          project=SAVE_DIR, name='test_eval', exist_ok=True)

test_results = {
    'mAP50':       float(metrics.box.map50),
    'mAP50_95':    float(metrics.box.map),
    'precision':   float(metrics.box.mp),
    'recall':      float(metrics.box.mr),
    'f1':          float(2 * metrics.box.mp * metrics.box.mr / (metrics.box.mp + metrics.box.mr + 1e-16)),
}
print('\n=== TEST SET METRİKLERİ ===')
for k, v in test_results.items():
    print(f'{k:12s}: {v:.4f}')

## 5. Tez Figürleri — Publication Quality

In [ ]:
import pandas as pd
import numpy as np

plt.rcParams.update({'font.size': 11, 'axes.grid': True, 'grid.alpha': 0.3})

df = pd.read_csv(f'{TRAIN_DIR}/results.csv')
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
loss_pairs = [
    ('train/box_loss', 'val/box_loss', 'Box Loss'),
    ('train/cls_loss', 'val/cls_loss', 'Classification Loss'),
    ('train/dfl_loss', 'val/dfl_loss', 'DFL Loss'),
]
for ax, (tr_col, va_col, title) in zip(axes, loss_pairs):
    if tr_col in df.columns: ax.plot(df['epoch'], df[tr_col], label='Eğitim', linewidth=2)
    if va_col in df.columns: ax.plot(df['epoch'], df[va_col], label='Doğrulama', linewidth=2, linestyle='--')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Kayıp'); ax.set_title(title); ax.legend()
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/loss_curves.pdf', bbox_inches='tight')
plt.savefig(f'{FIGURES_DIR}/loss_curves.png', bbox_inches='tight', dpi=200)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
if 'metrics/mAP50(B)' in df.columns:
    axes[0].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP@0.5', linewidth=2)
    axes[0].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95', linewidth=2, linestyle='--')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('mAP'); axes[0].set_title('Mean Average Precision'); axes[0].legend()
if 'metrics/precision(B)' in df.columns:
    axes[1].plot(df['epoch'], df['metrics/precision(B)'], label='Precision', linewidth=2)
    axes[1].plot(df['epoch'], df['metrics/recall(B)'],    label='Recall',    linewidth=2, linestyle='--')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Skor'); axes[1].set_title('Precision & Recall'); axes[1].legend()
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/metrics_curves.pdf', bbox_inches='tight')
plt.savefig(f'{FIGURES_DIR}/metrics_curves.png', bbox_inches='tight', dpi=200)
plt.show()

In [ ]:
import time
import seaborn as sns

test_dir = f'{DATASET_DIR}/test/images'
test_imgs_all = sorted(os.listdir(test_dir))
bench_imgs = test_imgs_all[:200]

for img_name in bench_imgs[:3]:
    best_model.predict(f'{test_dir}/{img_name}', verbose=False)

times_ms = []
for img_name in bench_imgs:
    t0 = time.perf_counter()
    best_model.predict(f'{test_dir}/{img_name}', verbose=False, imgsz=640)
    times_ms.append((time.perf_counter() - t0) * 1000)

times_arr = np.array(times_ms)
fps_stats = {
    'n_samples':     len(times_arr),
    'mean_ms':       float(times_arr.mean()),
    'median_ms':     float(np.median(times_arr)),
    'p95_ms':        float(np.percentile(times_arr, 95)),
    'p99_ms':        float(np.percentile(times_arr, 99)),
    'std_ms':        float(times_arr.std()),
    'mean_fps':      float(1000 / times_arr.mean()),
    'p95_fps':       float(1000 / np.percentile(times_arr, 95)),
}
print('\n=== INFERENCE BENCHMARK ===')
for k, v in fps_stats.items():
    print(f'{k:12s}: {v:.2f}')

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(times_arr, bins=40, kde=True, ax=ax, color='steelblue', edgecolor='black')
ax.axvline(times_arr.mean(),                 color='green',  linestyle='--', label=f'Ortalama: {times_arr.mean():.1f} ms')
ax.axvline(np.percentile(times_arr, 95),     color='orange', linestyle='--', label=f'P95: {np.percentile(times_arr, 95):.1f} ms')
ax.axvline(33.3,                             color='red',    linestyle=':',  label='30 FPS sınırı (33.3 ms)')
ax.set_xlabel('Inference süresi (ms/görüntü)'); ax.set_ylabel('Frekans')
ax.set_title(f'Plaka Detector — Inference Time Dağılımı (n={len(times_arr)})')
ax.legend(); plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/inference_time_hist.pdf', bbox_inches='tight')
plt.savefig(f'{FIGURES_DIR}/inference_time_hist.png', bbox_inches='tight', dpi=200)
plt.show()

In [ ]:
random.seed(42)
samples = random.sample(test_imgs_all, min(9, len(test_imgs_all)))

fig, axes = plt.subplots(3, 3, figsize=(16, 16))
for ax, name in zip(axes.flatten(), samples):
    img_path = f'{test_dir}/{name}'
    res = best_model.predict(img_path, conf=0.25, verbose=False)
    annotated = res[0].plot()
    n_det = len(res[0].boxes) if res[0].boxes is not None else 0
    avg_conf = float(res[0].boxes.conf.mean()) if n_det > 0 else 0.0
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{name[:30]}\nDetection: {n_det}, avg conf: {avg_conf:.2f}', fontsize=10)
    ax.axis('off')
plt.suptitle('Test Set — Örnek Prediction\'lar', fontsize=14, y=1.0)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/sample_predictions.pdf', bbox_inches='tight')
plt.savefig(f'{FIGURES_DIR}/sample_predictions.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
fail_candidates = []
for name in test_imgs_all:
    img_path = f'{test_dir}/{name}'
    res = best_model.predict(img_path, conf=0.1, verbose=False)
    n_det = len(res[0].boxes) if res[0].boxes is not None else 0
    max_conf = float(res[0].boxes.conf.max()) if n_det > 0 else 0.0
    if n_det == 0 or max_conf < 0.5:
        fail_candidates.append((name, n_det, max_conf))
    if len(fail_candidates) >= 9: break

if fail_candidates:
    fig, axes = plt.subplots(3, 3, figsize=(16, 16))
    for ax, (name, n_det, max_conf) in zip(axes.flatten(), fail_candidates):
        img_path = f'{test_dir}/{name}'
        res = best_model.predict(img_path, conf=0.1, verbose=False)
        annotated = res[0].plot()
        ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        reason = 'No detection' if n_det == 0 else f'Düşük conf: {max_conf:.2f}'
        ax.set_title(f'{name[:30]}\n{reason}', fontsize=10, color='darkred')
        ax.axis('off')
    for ax in axes.flatten()[len(fail_candidates):]: ax.axis('off')
    plt.suptitle(f'Failure Case Galerisi — {len(fail_candidates)} örnek', fontsize=14, y=1.0)
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/failure_cases.pdf', bbox_inches='tight')
    plt.savefig(f'{FIGURES_DIR}/failure_cases.png', bbox_inches='tight', dpi=150)
    plt.show()
    print(f'{len(fail_candidates)} failure case bulundu.')
else:
    print('Failure case bulunamadı — model çok iyi (veya test set küçük).')

## 6. Kayıt & Export

In [ ]:
import shutil, json
from datetime import datetime

FINAL_WEIGHT = f'{WEIGHTS_DIR}/plate.pt'
shutil.copy(BEST_PT, FINAL_WEIGHT)
size_mb = os.path.getsize(FINAL_WEIGHT) / 1024 / 1024
print(f'Plate detector: {FINAL_WEIGHT} ({size_mb:.2f} MB)')

thesis_metrics = {
    'meta': {
        'model':         'yolov8n',
        'dataset':       f'{ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT}/v{ROBOFLOW_VERSION}',
        'epochs':        80,
        'imgsz':         640,
        'batch':         32,
        'train_date':    datetime.utcnow().isoformat() + 'Z',
        'gpu':           torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
        'weight_size_mb': round(size_mb, 2),
    },
    'dataset': split_counts,
    'test_metrics': test_results,
    'inference_benchmark': fps_stats,
}
metrics_path = f'{METRICS_DIR}/thesis_metrics.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(thesis_metrics, f, indent=2, ensure_ascii=False)
print(f'\nTez metrikleri: {metrics_path}')
print(json.dumps(thesis_metrics, indent=2, ensure_ascii=False))

In [ ]:
from google.colab import files
files.download(FINAL_WEIGHT)
shutil.make_archive(f'{SAVE_DIR}/thesis_figures', 'zip', FIGURES_DIR)
files.download(f'{SAVE_DIR}/thesis_figures.zip')

## 7. Sonraki Adım

1. ✅ `weights/plate.pt`'i local'deki `hatched-area-violation-detection/weights/` klasörüne taşı
2. ✅ `thesis_figures.zip`'i `report/figures/` klasörüne aç
3. `src/plate/` modülü local'de hazır olduğunda end-to-end pipeline test edilecek
4. Drive üzerinde tüm artifact'lar `tez_models/plate_detector/` altında — yedeklendi